In [1]:
%%writefile server_app.py
import os
from fastapi import FastAPI
from langserve import add_routes
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

app = FastAPI(
    title="Smart Contract Summary & Q&A Assistant Server",
    description="An AI-powered server for analyzing smart contracts and documents using RAG logic.",
    version="1.0.0"
)

llm = ChatOpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN"),
    model="meta-llama/Llama-3.1-8B-Instruct:novita",
    temperature=0.4, 
    max_tokens=1000
)

system_instruction = (
    "You are a helpful and professional Assistant. "
    "1. GREETINGS: Be polite and answer general greetings (e.g., 'Hello', 'How are you?') naturally. "
    "2. CORE TASK: When asked about documents, use ONLY the provided context to answer. "
    "3. CITATIONS: Always mention the Page Number when extracting facts (e.g., 'As per Page X...'). "
    "4. GUARD-RAILS: If a question is NOT in the document and NOT a greeting, say you don't have that info. "
    "5. OUTPUT FORMAT: Answer concisely followed by a 'Sources' section.\n\n"
    "CONTEXT:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()
add_routes(app, chain, path="/rag_chat")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=9017)

Overwriting server_app.py
